# Phase 2a - DarkIR Exploration

Before writing the real DarkIR-lite training script, confirm the facts we don't yet know about the official DarkIR repo (github.com/cidautai/DarkIR, MIT license) and its `Cidaut/DarkIR` HuggingFace checkpoint:

1. Does Kaggle's preinstalled torch/CUDA match what DarkIR expects (repo pins torch 2.5.1 / CUDA 12.4)?
2. What's the actual model class name and constructor signature in `archs/`?
3. What's the actual DarkIR-m checkpoint filename on HuggingFace, and does `load_state_dict` succeed cleanly?
4. What input convention (shape, normalization) does the model expect?
5. Does a forward pass work end-to-end on a real (degraded) EndoSLAM frame?

**GPU is off** -- this is exploration only, no training happens here, no need to burn GPU quota. Findings feed directly into `src/darkir_lite/model.py` + `train.py`, which get written *after* this notebook confirms the real API -- same probe-first pattern that got Phase 1's dataset loader right after its folder-structure guesses were wrong.

Cells here are defensive (try/except + prints) rather than hard-asserting, since the goal is gathering facts, not pass/fail.

## 0. Setup: clone our repo + DarkIR

In [ ]:
REPO_URL = "https://github.com/ritiksharma3/endoslam.git"
DARKIR_URL = "https://github.com/cidautai/DarkIR.git"

!git clone $REPO_URL repo
!git clone $DARKIR_URL repo/DarkIR_upstream

import subprocess
darkir_sha = subprocess.run(
    ["git", "-C", "repo/DarkIR_upstream", "rev-parse", "HEAD"],
    capture_output=True, text=True
).stdout.strip()
print(f"DarkIR commit SHA (pin this once confirmed working): {darkir_sha}")

%cd repo
!pip install -q -r environment/requirements.txt

import sys
sys.path.insert(0, ".")
sys.path.insert(0, "DarkIR_upstream")

## 1. torch / CUDA compatibility check

In [ ]:
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("torch CUDA build:", torch.version.cuda)
print("DarkIR repo pins: torch 2.5.1 / CUDA 12.4 -- compare above, flag if wildly different")

try:
    import huggingface_hub
    print("huggingface_hub already available:", huggingface_hub.__version__)
except ImportError:
    print("huggingface_hub NOT available -- installing")
    !pip install -q huggingface_hub
    import huggingface_hub
    print("huggingface_hub installed:", huggingface_hub.__version__)

# Confirmed via local static inspection of the cloned repo: archs/__init__.py does
# `from ptflops import get_model_complexity_info` at module level, which breaks
# `from archs.DarkIR import DarkIR` entirely if ptflops isn't installed (this is
# what failed run 1). It's a small FLOPs-counting utility, safe to add.
try:
    import ptflops
    print("ptflops already available:", ptflops.__version__)
except ImportError:
    print("ptflops NOT available -- installing (required just to import archs/)")
    !pip install -q ptflops
    import ptflops
    print("ptflops installed:", ptflops.__version__)

## 2. Inspect archs/ for the real model class

In [ ]:
import os

archs_dir = "DarkIR_upstream/archs"
print("archs/ contents:", sorted(os.listdir(archs_dir)) if os.path.isdir(archs_dir) else "NOT FOUND")

print("\n--- class definitions found in archs/*.py ---")
for fname in sorted(os.listdir(archs_dir)):
    if not fname.endswith(".py"):
        continue
    path = os.path.join(archs_dir, fname)
    with open(path, encoding="utf-8") as f:
        for i, line in enumerate(f):
            if line.startswith("class "):
                print(f"{fname}:{i+1}: {line.strip()}")

In [ ]:
# Confirmed via local static inspection: class is archs.DarkIR.DarkIR, constructor
# is __init__(self, img_channel=3, width=32, middle_blk_num_enc=2, middle_blk_num_dec=2,
# enc_blk_nums=[1,2,3], dec_blk_nums=[3,1,1], dilations=[1,4,9], extra_depth_wise=True).
# Re-confirm here against Kaggle's actual environment (torch 2.10 vs their pinned 2.5.1).
import inspect

from archs.DarkIR import DarkIR as DarkIRModel
print("import archs.DarkIR.DarkIR succeeded")
print(inspect.signature(DarkIRModel.__init__))

## 3. Find and download the DarkIR-m checkpoint from HuggingFace

In [ ]:
from huggingface_hub import list_repo_files, hf_hub_download

HF_REPO = "Cidaut/DarkIR"
try:
    files = list_repo_files(HF_REPO)
    print(f"Files in {HF_REPO}:")
    for f in files:
        print(" ", f)
except Exception as e:
    print(f"list_repo_files failed: {type(e).__name__}: {e}")
    files = []

In [ ]:
# Confirmed via local inspection of options/test/LOLBlur.yml: the HF repo's only
# checkpoint, DarkIR_384.pt, is the width=32 variant (exactly what we want).
CHECKPOINT_FILENAME = "DarkIR_384.pt"
print("checkpoint-like files on HF:", [f for f in files if f.endswith((".pth", ".ckpt", ".pt"))])
assert CHECKPOINT_FILENAME in files, f"expected {CHECKPOINT_FILENAME} in HF repo file list, got {files}"

checkpoint_path = None
try:
    checkpoint_path = hf_hub_download(repo_id=HF_REPO, filename=CHECKPOINT_FILENAME)
    print("downloaded to:", checkpoint_path)
except Exception as e:
    print(f"hf_hub_download failed: {type(e).__name__}: {e}")

## 4. Instantiate DarkIR-m and load the checkpoint

In [ ]:
# width=32 per this project's decision (0.5x of DarkIR-l's width=64).
# Param name is a guess ("width") -- adjust based on the signature printed in section 2.
model = None
try:
    model = DarkIRModel(width=32)
    print("model instantiated:", sum(p.numel() for p in model.parameters()), "params")
except Exception as e:
    print(f"instantiation failed: {type(e).__name__}: {e}")

In [ ]:
# Confirmed via local inspection of testing.py / inference.py's load_model():
# checkpoint is torch.load(...)['params'] -- NOT 'state_dict' or 'model_state_dict'.
# Their loader adds a "module." prefix because create_model() wraps in DDP; we're
# not using DDP for a single-GPU Kaggle session, so load 'params' directly, unprefixed.
if model is not None and checkpoint_path is not None:
    checkpoints = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    print("checkpoint top-level keys:", list(checkpoints.keys()) if isinstance(checkpoints, dict) else type(checkpoints))
    weights = checkpoints["params"]
    result = model.load_state_dict(weights, strict=False)
    print("missing keys:", len(result.missing_keys), result.missing_keys[:5])
    print("unexpected keys:", len(result.unexpected_keys), result.unexpected_keys[:5])
else:
    print("skipped -- model or checkpoint_path not available, see earlier cells")

## 5. Check inference.py for the expected input convention

In [ ]:
# Confirmed via local inspection of inference.py: path_to_tensor() uses
# PIL Image.open(...).convert('RGB') + transforms.ToTensor() -- i.e. plain [0,1]
# float RGB, CHW, no extra mean/std normalization. Matches our own
# dark_degradation.py convention exactly. pad_tensor() pads H/W to a multiple of 8
# (our 320x320 EndoSLAM frames are already divisible by 8, so no padding needed).
# Print the actual snippet here to re-confirm against Kaggle's checked-out copy.
inf_path = "DarkIR_upstream/inference.py"
with open(inf_path, encoding="utf-8") as f:
    content = f.read()
start = content.find("def path_to_tensor")
end = content.find("def pad_tensor") + 400
print(content[start:end])

## 6. Forward pass: dummy tensor, then a real degraded EndoSLAM frame

In [ ]:
if model is not None:
    model.eval()
    dummy = torch.randn(1, 3, 256, 256)
    try:
        with torch.no_grad():
            out = model(dummy)
        out_shape = out.shape if hasattr(out, "shape") else [o.shape for o in out]
        print("dummy forward pass output shape:", out_shape)
    except Exception as e:
        print(f"dummy forward pass failed: {type(e).__name__}: {e}")
else:
    print("skipped -- no model")

In [ ]:
import yaml
import os as _os

def find_endoslam_root(base="/kaggle/input", max_depth=4):
    for root, dirs, _files in _os.walk(base):
        depth = root[len(base):].count(_os.sep)
        if depth > max_depth:
            dirs[:] = []
            continue
        if _os.path.basename(root).lower() == "endoslam":
            return root
    return None

with open("configs/config.yaml") as f:
    config = yaml.safe_load(f)
config["data"]["root"] = find_endoslam_root()

from src.data.endoslam_dataset import EndoSLAMStomachDataset
from src.data.dark_degradation import degrade_frame

all_cameras = [config["data"]["synthetic_cam"]] + config["data"]["real_cams"]
ds = EndoSLAMStomachDataset(config, split="train", cameras=all_cameras,
                             context_window=config["reconstruction"]["context_window"])
frames = ds.flatten_for_enhancement()
print(f"{len(frames)} frames available via flatten_for_enhancement()")

import cv2
import numpy as np
sample = frames[0]
img = cv2.imread(sample.image_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
dark = degrade_frame(img)
print("clean shape:", img.shape, "dark shape:", dark.shape)

if model is not None:
    tensor = torch.from_numpy(dark).permute(2, 0, 1).unsqueeze(0).float()
    try:
        with torch.no_grad():
            out = model(tensor)
        out_shape = out.shape if hasattr(out, "shape") else [o.shape for o in out]
        print("real-frame forward pass output shape:", out_shape)
    except Exception as e:
        print(f"real-frame forward pass failed: {type(e).__name__}: {e}")

## Done

Collect the findings from each section above (real class name/signature, checkpoint filename, missing/unexpected key counts, input convention, forward-pass shapes) and write them into `PROGRESS.md`, then implement `src/darkir_lite/model.py` + `train.py` against these confirmed facts -- don't guess further locally.